






















## Seguim intentant trobar una bona distribució de T respecte TSV i TCV

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path


# Put your notebook path here once (or set it via VS Code explorer)
NOTEBOOK_DIR = Path(r"C:\Users\labor\Documents\GitHub\TFM-Thermal-Walks\statistical_try")  # <-- change this

out_dir = NOTEBOOK_DIR / "figures"
out_dir.mkdir(parents=True, exist_ok=True)




In [ ]:
df_votes = pd.read_csv(r"../../data/all_surveys(votes).csv")
df_votes["<T-T_fixed>"]
print(df_votes.columns)



## Primer farem un scatter de TSV (T)

In [ ]:
import matplotlib.pyplot as plt



# choose the two columns
x_col = "<T-T_fixed+<T>>"
y_col = "thermal_sensation"
tsv_map = {
    "Cool": 1,
    "Slightly cool": 2,
    "Neutral": 3,
    "Slightly warm": 4,
    "Warm": 5,
    "Hot": 6,
    "Very hot": 7
}


df_plot = df_votes.copy()
df_plot["tsv_num"] = df_plot[y_col].map(tsv_map)
df_plot[x_col] = pd.to_numeric(df_plot[x_col], errors="coerce")
df_plot = df_plot.dropna(subset=[x_col, "tsv_num"]).sort_values(x_col)

ax = df_plot.plot.scatter(x=x_col, y="tsv_num")

# y-axis: ordered 1..7, but show labels (not numbers)
order = [k for k, v in sorted(tsv_map.items(), key=lambda kv: kv[1])]
ax.set_yticks(range(1, 8))
ax.set_yticklabels(order)
ax.set_ylim(0.5, 7.5)

ax.set_ylabel("Thermal Sensation Vote (Cold to Hot)")
ax.set_xlabel("Corrected Temperature (°C)")
plt.show()

## No ens ensenya absolutament res tot i que sembla prou coherent? Molaria centrar-ho amb $\Delta T/T$ del thermal walk

# MEAN T x TSV-> Ara MEAN T. Després farem al revés (MEAN TSV(t BIN))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

x_col = "<HDX-HDX_fixed+<HDX>>"
y_col = "thermal_sensation"

# keep only needed columns and drop missing values
plot_df = df_votes[[x_col, y_col]].dropna().copy()

# ensure numeric x
plot_df = plot_df.dropna(subset=[x_col])

# map thermal sensation to numeric TSV (1..7)
plot_df["TSV"] = plot_df[y_col].map(tsv_map)
plot_df = plot_df.dropna(subset=["TSV"])

# group by category and compute temperature statistics
summary = (
    plot_df.groupby([y_col, "TSV"], as_index=False)[x_col]
    .agg(mean_T="mean", std_T="std", n="count")
    .sort_values("TSV")
)

# standard error
summary["sem_T"] = summary["std_T"] / summary["n"].pow(0.5)

print(summary)

# Plot: mean temperature vs TSV category, with horizontal error bars
plt.figure(figsize=(7, 5))
plt.errorbar(
    summary["mean_T"],
    summary["TSV"],
    xerr=summary["sem_T"],  # or summary["std_T"]
    fmt="o",
    capsize=4,
)

# y axis ordered 1..7 with labels (not numbers)
ordered_labels = [k for k, v in sorted(tsv_map.items(), key=lambda kv: kv[1])]
plt.yticks(range(1, 8), ordered_labels)
plt.ylim(0.5, 7.5)

plt.xlabel("Corrected Temperature (°C)")
plt.ylabel("Thermal Sensation")
plt.title("Temperature vs Thermal Sensation")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

x_col = "<HDX-HDX_fixed+<HDX>>"
y_col = "thermal_sensation"

# Recode categories
sensation_map = {
    "Cool": "Cool / Slightly cool",
    "Slightly cool": "Cool / Slightly cool",
    "Neutral": "Neutral",
    "Slightly warm": "Slightly warm",
    "Warm": "Warm",
    "Hot": "Hot" ,
    "Very hot": "Very hot",
}

# Ordered positions for plotting
tsv_group_map = {
    "Cool / Slightly cool": 1,
    "Neutral": 2,
    "Slightly warm": 3,
    "Warm": 4,
    "Hot": 5,
    "Very hot": 6,
}

# keep only needed columns and drop missing values
plot_df = df_votes[[x_col, y_col]].dropna().copy()

# ensure numeric x
plot_df[x_col] = pd.to_numeric(plot_df[x_col], errors="coerce")
plot_df = plot_df.dropna(subset=[x_col])

# recode thermal sensation
plot_df["TSV_group"] = plot_df[y_col].map(sensation_map)
plot_df = plot_df.dropna(subset=["TSV_group"])

# numeric plotting position
plot_df["TSV_num"] = plot_df["TSV_group"].map(tsv_group_map)

# group by new category and compute humidex statistics
summary = (
    plot_df.groupby(["TSV_group", "TSV_num"], as_index=False)[x_col]
    .agg(mean_HDX="mean", std_HDX="std", n="count")
    .sort_values("TSV_num")
)

# standard error
summary["sem_HDX"] = summary["std_HDX"] / summary["n"].pow(0.5)

print(summary)

# Plot
plt.figure(figsize=(7, 5))
plt.errorbar(
    summary["mean_HDX"],
    summary["TSV_num"],
    xerr=summary["sem_HDX"],
    fmt="o",
    capsize=4,
)

ordered_labels = [k for k, v in sorted(tsv_group_map.items(), key=lambda kv: kv[1])]
plt.yticks(range(1, len(ordered_labels) + 1), ordered_labels)
plt.ylim(0.5, len(ordered_labels) + 0.5)

plt.xlabel("Corrected Humidex")
plt.ylabel("Thermal Sensation")
plt.title("Humidex vs Thermal Sensation")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ----------------------------
# Base columns
# ----------------------------
x_col = "<T-T_fixed+<T>>"
y_col = "thermal_sensation"
gender_col = "gender"


tsv_map = {
    "Cool": 1,
    "Slightly cool": 2,
    "Neutral": 3,
    "Slightly warm": 4,
    "Warm": 5,
    "Hot": 6,
    "Very hot": 7
}

# ----------------------------
# Prepare dataframe
# ----------------------------
plot_df = df_votes[[x_col, y_col, gender_col]].dropna().copy()
plot_df[x_col] = pd.to_numeric(plot_df[x_col], errors="coerce")
plot_df["TSV"] = plot_df[y_col].map(tsv_map)
plot_df = plot_df.dropna(subset=[x_col, "TSV"]).copy()
plot_df["TSV"] = plot_df["TSV"].astype(int)

# ----------------------------
# Summaries by gender
# ----------------------------
def summarize_by_gender(df_in: pd.DataFrame, gender: str) -> pd.DataFrame:
    s = (
        df_in[df_in[gender_col] == gender]
        .groupby(["TSV"], as_index=False)[x_col]
        .agg(mean_T="mean", std_T="std", n="count")
        .sort_values("TSV")
    )
    s["sem_T"] = s["std_T"] / np.sqrt(s["n"])
    s["ci95_T"] = 1.95 * s["sem_T"]  # keep your 1.95 factor
    s["gender"] = gender
    return s

genders = ["Woman", "Man"]  # change to plot_df[gender_col].dropna().unique() for all
summaries = []
for g in genders:
    if (plot_df[gender_col] == g).any():
        summaries.append(summarize_by_gender(plot_df, g))

summary_all = pd.concat(summaries, ignore_index=True) if summaries else pd.DataFrame()
print(summary_all)

# ----------------------------
# Plot: mean temperature vs TSV (3-level) with 95% CI, by gender
# ----------------------------
plt.figure(figsize=(7, 5))

for g in genders:
    s = summary_all[summary_all["gender"] == g]
    if s.empty:
        continue
    plt.errorbar(
        s["mean_T"],
        s["TSV"],
        xerr=s["ci95_T"],
        fmt="o",
        capsize=4,
        label=g,
    )

plt.yticks([1, 2, 3, 4, 5, 6, 7], ["Cool", "Slightly cool", "Neutral", "Slightly warm", "Warm", "Hot", "Very hot"])
plt.xlabel("Corrected Temperature (°C)")
plt.ylabel("TSV")
plt.title("Temperature vs grouped TSV (95% CI), by gender")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Probem amb tres grups i després també podem fer-ho amb TCV

In [ ]:
df_votes = pd.read_csv(r"../../data/all_surveys(votes).csv")


# choose the two columns
x_col = "<T-T_fixed+<T>>"
y_col = "thermal_sensation"
tsv_map = {
    "Cool": 1,
    "Slightly cool": 1,
    "Neutral": 2,
    "Slightly warm": 2,
    "Warm": 3,
    "Hot": 3,
    "Very hot": 3
}


# keep only needed columns and drop missing values
plot_df = df_votes[[x_col, y_col]].dropna().copy()

# map thermal sensation to numeric TSV
plot_df["TSV"] = plot_df[y_col].map(tsv_map)

# group by sensation / TSV and compute temperature statistics
summary = (
    plot_df.groupby(["TSV"], as_index=False)[x_col]
    .agg(mean_T="mean", std_T="std", n="count")
    .sort_values("TSV")
)

# optional: standard error instead of standard deviation
summary["sem_T"] = summary["std_T"] / summary["n"]**0.5

print(summary)

# Plot: mean temperature vs mean TSV category, with horizontal error bars in T
plt.figure(figsize=(7, 5))
plt.errorbar(
    summary["mean_T"],          # x = mean temperature
    summary["TSV"],             # y = thermal sensation vote
    xerr=1.95*summary["sem_T"],      # or use summary["std_T"]
    fmt="o",
    capsize=4
)

summary

plt.yticks(summary["TSV"])
plt.xlabel("Corrected Temperature (°C)")
plt.ylabel("Mean TSV")
plt.title("Temperature vs Thermal Sensation (95% CI)")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ----------------------------
# Base columns
# ----------------------------
x_col = "<T-T_fixed+<T>>"
y_col = "thermal_sensation"
gender_col = "gender"

# 3-level TSV mapping (your grouping)


# ----------------------------
# Prepare dataframe
# ----------------------------
plot_df = df_votes[[x_col, y_col, gender_col]].dropna().copy()
plot_df[x_col] = pd.to_numeric(plot_df[x_col], errors="coerce")
plot_df["TSV"] = plot_df[y_col].map(tsv_map)
plot_df = plot_df.dropna(subset=[x_col, "TSV"]).copy()
plot_df["TSV"] = plot_df["TSV"].astype(int)

# ----------------------------
# Summaries by gender
# ----------------------------
def summarize_by_gender(df_in: pd.DataFrame, gender: str) -> pd.DataFrame:
    s = (
        df_in[df_in[gender_col] == gender]
        .groupby(["TSV"], as_index=False)[x_col]
        .agg(mean_T="mean", std_T="std", n="count")
        .sort_values("TSV")
    )
    s["sem_T"] = s["std_T"] / np.sqrt(s["n"])
    s["ci95_T"] = 1.95 * s["sem_T"]  # keep your 1.95 factor
    s["gender"] = gender
    return s

genders = ["Woman", "Man"]  # change to plot_df[gender_col].dropna().unique() for all
summaries = []
for g in genders:
    if (plot_df[gender_col] == g).any():
        summaries.append(summarize_by_gender(plot_df, g))

summary_all = pd.concat(summaries, ignore_index=True) if summaries else pd.DataFrame()
print(summary_all)

# ----------------------------
# Plot: mean temperature vs TSV (3-level) with 95% CI, by gender
# ----------------------------
plt.figure(figsize=(7, 5))

for g in genders:
    s = summary_all[summary_all["gender"] == g]
    if s.empty:
        continue
    plt.errorbar(
        s["mean_T"],
        s["TSV"],
        xerr=s["sem_T"],
        fmt="o",
        capsize=4,
        label=g,
    )

plt.yticks([1, 2, 3], ["Cool", "Neutral", "Warm/Hot"])
plt.xlabel("Corrected Temperature (°C)")
plt.ylabel("TSV (grouped)")
plt.title("Temperature vs grouped TSV (sem), by gender")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:

plt.figure(figsize=(7, 5))

for g in genders:
    s = summary_all[summary_all["gender"] == g]
    if s.empty:
        continue
    plt.errorbar(
        s["mean_T"],
        s["TSV"],
        xerr=s["ci95_T"],
        fmt="o",
        capsize=4,
        label=g,
    )

plt.yticks([1, 2, 3], ["Cool", "Neutral", "Warm/Hot"])
plt.xlabel("Corrected Temperature (°C)")
plt.ylabel("TSV (grouped)")
plt.title("Temperature vs grouped TSV (95% CI), by gender")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))
plt.errorbar(summary["mean_T"], summary["TSV"], xerr=summary["sem_T"], fmt="o", capsize=4)
plt.xlabel("Corrected Temperature (°C)")
plt.ylabel("TSV")
plt.title("Mean temperature with SEM")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(7, 5))
plt.errorbar(summary["mean_T"], summary["TSV"], xerr=summary["std_T"], fmt="o", capsize=4)
plt.xlabel("Corrected Temperature (°C)")
plt.ylabel("TSV")
plt.title("Mean temperature with SD")
plt.grid(True, alpha=0.3)
plt.show()

## Nem a mirar lo de fer $\frac{\Delta T}{T}$ k és igual a $\Delta T$

Mirem fent la relativa respecte el total i respecte la mitjana de cada caminada (aquesta te més sentit per a veure)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# Base columns
# ----------------------------
temp_col = "<T-T_fixed+<T>>"
y_col = "thermal_sensation"

tsv_map = {
    "Cool": 1,
    "Slightly cool": 2,
    "Neutral": 3,
    "Slightly warm": 4,
    "Warm": 5,
    "Hot": 6,
    "Very hot": 7,
}

# ----------------------------
# Prep + Kelvin temperature
# ----------------------------
df = df_votes.copy()
df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")
df["walk"] = df["space_code"].astype(str).str[:2]
df["T_K"] = df[temp_col] + 273.15

# ----------------------------
# Relative temperatures (ONLY relatives)
# (use the standard definition: (T - mean)/mean
# ----------------------------
df["T_mean_walk_K"] = df.groupby("walk")["T_K"].transform("mean")
df["relative_temperature_walk_K"] = (df["T_K"] - df["T_mean_walk_K"])

T_mean_all_K = df["T_K"].mean(skipna=True)
df["relative_temperature_all_K"] = (df["T_K"] - T_mean_all_K) 

# ----------------------------
# Plotting dataframe
# ----------------------------
plot_df = df[["relative_temperature_walk_K", "relative_temperature_all_K", y_col]].dropna().copy()
plot_df["TSV"] = plot_df[y_col].map(tsv_map)
plot_df = plot_df.dropna(subset=["TSV"]).copy()
plot_df["TSV"] = plot_df["TSV"].astype(int)

y_ticks = list(range(1, 8))
y_labels = [k for k, v in sorted(tsv_map.items(), key=lambda kv: kv[1])]

def make_summary(df_in: pd.DataFrame, x_rel_col: str) -> pd.DataFrame:
    s = (
        df_in.groupby("TSV", as_index=False)[x_rel_col]
        .agg(mean_x="mean", std_x="std", n="count")
        .sort_values("TSV")
    )
    s["sem_x"] = s["std_x"] / np.sqrt(s["n"])
    return s

def plot_total_and_mean(df_in: pd.DataFrame, x_rel_col: str, title_prefix: str, x_label: str):
    # Total scatter
    plt.figure(figsize=(7, 5))
    plt.scatter(df_in[x_rel_col], df_in["TSV"], alpha=0.35)
    plt.yticks(y_ticks, y_labels)
    plt.ylim(0.5, 7.5)
    plt.xlabel(x_label)
    plt.ylabel("Thermal Sensation")
    plt.title(f"{title_prefix} — total")
    plt.grid(True, alpha=0.3)
    plt.show()

    # Mean per TSV with SEM (horizontal errorbars)
    summary = make_summary(df_in, x_rel_col)

    plt.figure(figsize=(7, 5))
    plt.errorbar(
        summary["mean_x"],
        summary["TSV"],
        xerr=summary["sem_x"],
        fmt="o",
        capsize=4,
    )
    plt.yticks(y_ticks, y_labels)
    plt.ylim(0.5, 7.5)
    plt.xlabel(x_label)
    plt.ylabel("Thermal Sensation")
    plt.title(f"{title_prefix} — mean ± SEM")
    plt.grid(True, alpha=0.3)
    plt.show()

    return summary

# ----------------------------
# Plot both: total + mean (like before)
# ----------------------------
summary_walk = plot_total_and_mean(
    plot_df,
    "relative_temperature_walk_K",
    "TSV vs centered temperature (scaled by walk mean)",
    "Relative temperature: (T_K - mean_walk)",
)

summary_all = plot_total_and_mean(
    plot_df,
    "relative_temperature_all_K",
    "TSV vs centered temperature (scaled by global mean)",
    "Relative temperature: (T_K - mean_all)",
)

print("=== Summary (relative by walk) ===")
print(summary_walk)

print("\n=== Summary (relative by all) ===")
print(summary_all)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# Base columns
# ----------------------------
temp_col = "<T-T_fixed+<T>>"
y_col = "thermal_sensation"
gender_col = "gender"

tsv_map = {
    "Cool": 1,
    "Slightly cool": 2,
    "Neutral": 3,
    "Slightly warm": 4,
    "Warm": 5,
    "Hot": 6,
    "Very hot": 7,
}

# ----------------------------
# Prep + Kelvin temperature (centered by walk mean)
# ----------------------------
df = df_votes.copy()
df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")
df["walk"] = df["space_code"].astype(str).str[:2]
df["T_K"] = df[temp_col] + 273.15

df["T_mean_walk_K"] = df.groupby("walk")["T_K"].transform("mean")
df["T_centered_walk_K"] = df["T_K"] - df["T_mean_walk_K"]  # this is what your code is actually doing

df["TSV"] = df[y_col].map(tsv_map)

plot_df = df[["T_centered_walk_K", "TSV", gender_col]].dropna().copy()
plot_df["TSV"] = plot_df["TSV"].astype(int)

y_ticks = list(range(1, 8))
y_labels = [k for k, v in sorted(tsv_map.items(), key=lambda kv: kv[1])]

def make_summary(df_in: pd.DataFrame, x_col: str) -> pd.DataFrame:
    s = (
        df_in.groupby("TSV", as_index=False)[x_col]
        .agg(mean_x="mean", std_x="std", n="count")
        .sort_values("TSV")
    )
    s["sem_x"] = s["std_x"] / np.sqrt(s["n"])
    return s

# ----------------------------
# Plot: mean centered temperature for each TSV, by gender
# (ONLY the means plot)
# ----------------------------
plt.figure(figsize=(8, 5))

for g in ["Woman", "Man"]:  # change to plot_df[gender_col].unique() for all
    sub = plot_df[plot_df[gender_col] == g].copy()
    if sub.empty:
        continue

    summary = make_summary(sub, "T_centered_walk_K")
    plt.errorbar(
        summary["mean_x"],
        summary["TSV"],
        xerr=summary["sem_x"],
        fmt="o",
        capsize=4,
        label=g,
    )

plt.axvline(0, linewidth=1, alpha=0.5)
plt.yticks(y_ticks, y_labels)
plt.ylim(0.5, 7.5)
plt.xlabel("Temperature centered by walk mean (K)")
plt.ylabel("Thermal Sensation")
plt.title("Mean centered temperature by TSV, split by gender (± SEM)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Sembla que els efectes per persona (amb T centrada canvien). Llavors potser canvia per la caminada on estem? i no son reals

Mirem vots per categoria

In [ ]:
import pandas as pd

y_col = "thermal_sensation"
gender_col = "gender"

target = ["Cool", "Slightly cool","Neutral", "Very hot"]

counts = (
    df_votes[df_votes[gender_col].isin(["Woman", "Man"]) & df_votes[y_col].isin(target)]
    .groupby([gender_col, y_col])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=target, fill_value=0)
)

print(counts)

Viem que la variabilitat és important però tampoc massa boja

In [ ]:
import numpy as np
import pandas as pd

temp_col = "<HDX-HDX_fixed+<HDX>>"
y_col = "thermal_sensation"
gender_col = "gender"
sun_col = "sun"
wind_col = "wind"

wind_map = {
    "A strong wind": -1,
    "A moderate wind": -1,
    "A light wind": 0,
    "A very light wind": 0,
    "It's not windy": 1,
}

sun_map = {
    "In full shade": -1,
    "In a mixture of sun and shadow": 0,
    "In full sun": 1,
}

target_cats = ["Cool", "Slightly cool", "Neutral", "Slightly warm", "Warm", "Hot", "Very hot"]

df = df_votes.copy()
df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")
df["sun_num"] = df[sun_col].map(sun_map)
df["wind_num"] = df[wind_col].map(wind_map)

df = df[df[gender_col].isin(["Woman", "Man"]) & df[y_col].isin(target_cats)].copy()
df = df.dropna(subset=[temp_col, "sun_num", "wind_num", gender_col, y_col])

summary = (
    df.groupby([gender_col, y_col], as_index=False)
      .agg(
          n=("ID", "count") if "ID" in df.columns else (temp_col, "count"),
          mean_T=(temp_col, "mean"),
          mean_sun=("sun_num", "mean"),
          mean_wind=("wind_num", "mean"),
      )
)

# nicer ordering
summary[y_col] = pd.Categorical(summary[y_col], categories=target_cats, ordered=True)
summary = summary.sort_values([y_col, gender_col])

print(summary)

# Optional: show Woman-Man differences within each category
pivot = summary.pivot(index=y_col, columns=gender_col, values=["mean_T", "mean_sun", "mean_wind", "n"])
if ("Woman" in pivot["mean_T"].columns) and ("Man" in pivot["mean_T"].columns):
    diffs = pd.DataFrame({
        "ΔT_Woman-Man": pivot["mean_T"]["Woman"] - pivot["mean_T"]["Man"],
        "Δsun_Woman-Man": pivot["mean_sun"]["Woman"] - pivot["mean_sun"]["Man"],
        "Δwind_Woman-Man": pivot["mean_wind"]["Woman"] - pivot["mean_wind"]["Man"],
        "n_Woman": pivot["n"]["Woman"],
        "n_Man": pivot["n"]["Man"],
    })
    print("\n=== Woman - Man differences (within category) ===")
    print(diffs)

In [ ]:
import numpy as np
import pandas as pd

try:
    from scipy.stats import mannwhitneyu
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False

temp_col = "<T-T_fixed+<T>>"
gender_col = "gender"
y_col = "thermal_sensation"

sun_map = {"In full shade":-1, "In a mixture of sun and shadow":0, "In full sun":1}
wind_map = {"A strong wind":-1, "A moderate wind":-1, "A light wind":0, "A very light wind":0, "It's not windy":1}

target_cats = ["Slightly cool", "Cool", "Very hot"]

df = df_votes.copy()
df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")
df["sun_num"] = df["sun"].map(sun_map)
df["wind_num"] = df["wind"].map(wind_map)

df = df[df[gender_col].isin(["Woman","Man"]) & df[y_col].isin(target_cats)].dropna(
    subset=[temp_col, "sun_num", "wind_num", gender_col, y_col]
).copy()

def med_iqr(s):
    q1 = s.quantile(0.25)
    q2 = s.quantile(0.50)
    q3 = s.quantile(0.75)
    return q2, (q1, q3)

rows = []
for cat in target_cats:
    sub = df[df[y_col] == cat]
    for var, name in [(temp_col, "T"), ("sun_num", "sun"), ("wind_num", "wind")]:
        w = sub[sub[gender_col] == "Woman"][var]
        m = sub[sub[gender_col] == "Man"][var]
        if len(w) < 5 or len(m) < 5:
            continue

        w_med, (w_q1, w_q3) = med_iqr(w)
        m_med, (m_q1, m_q3) = med_iqr(m)

        out = {
            "category": cat,
            "var": name,
            "n_woman": len(w),
            "n_man": len(m),
            "woman_median": w_med,
            "man_median": m_med,
            "median_diff_W-M": w_med - m_med,
            "woman_IQR": (w_q1, w_q3),
            "man_IQR": (m_q1, m_q3),
        }

        if HAVE_SCIPY:
            stat, p = mannwhitneyu(w, m, alternative="two-sided")
            out["mw_p"] = p

        rows.append(out)

res = pd.DataFrame(rows)
print(res)

# AIXÒ EN TEORIA EM DIU QUE NO HI HA DIFERÈNCIES SIGNIFICATIVES ENTRE ELS DIES I LES CAMINADES MESURADES PER HOMES/DONES

In [ ]:
import numpy as np
import pandas as pd

temp_col = "<T-T_fixed+<T>>"
gender_col = "gender"
sun_col = "sun"     # change if different
wind_col = "wind"   # change if different

wind_map = {
    "A strong wind": -1,
    "A moderate wind": -1,
    "A light wind": 0,
    "A very light wind": 0,
    "It's not windy": 1,
}

sun_map = {
    "In full shade": -1,
    "In a mixture of sun and shadow": 0,
    "In full sun": 1,
}

df = df_votes.copy()

# Numeric conversions / mappings
df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")
df["sun_num"] = df[sun_col].map(sun_map)
df["wind_num"] = df[wind_col].map(wind_map)

# Keep Woman/Man only (edit if you want to include others)
df = df[df[gender_col].isin(["Woman", "Man"])].copy()

# Drop missing needed fields
df = df.dropna(subset=[temp_col, "sun_num", "wind_num", gender_col]).copy()

# =========================================================
# A) Vote-weighted (each row counts)
# =========================================================
vote_weighted = (
    df.groupby(gender_col, as_index=False)
      .agg(
          n_votes=(temp_col, "count"),
          mean_T=(temp_col, "mean"),
          mean_sun=("sun_num", "mean"),
          mean_wind=("wind_num", "mean"),
      )
)

print("=== Vote-weighted averages (each row = 1) ===")
print(vote_weighted)

if set(vote_weighted[gender_col]) >= {"Woman", "Man"}:
    vw = vote_weighted.set_index(gender_col)
    print("\nVote-weighted Woman - Man:")
    print(pd.Series({
        "ΔT (°C)": vw.loc["Woman", "mean_T"] - vw.loc["Man", "mean_T"],
        "Δsun index": vw.loc["Woman", "mean_sun"] - vw.loc["Man", "mean_sun"],
        "Δwind index": vw.loc["Woman", "mean_wind"] - vw.loc["Man", "mean_wind"],
    }))

# =========================================================
# B) Person-weighted (each person counts once) IF you have an id
# =========================================================
# Change this to your real column name if it exists:
possible_ids = [c for c in ["person_id", "participant_id", "user_id", "respondent_id", "id"] if c in df.columns]
person_id_col = possible_ids[0] if possible_ids else None

if person_id_col:
    person_means = (
        df.groupby([person_id_col, gender_col], as_index=False)
          .agg(
              person_mean_T=(temp_col, "mean"),
              person_mean_sun=("sun_num", "mean"),
              person_mean_wind=("wind_num", "mean"),
              n_votes_person=(temp_col, "count"),
          )
    )

    person_weighted = (
        person_means.groupby(gender_col, as_index=False)
          .agg(
              n_people=(person_id_col, "nunique"),
              mean_T=("person_mean_T", "mean"),
              mean_sun=("person_mean_sun", "mean"),
              mean_wind=("person_mean_wind", "mean"),
              median_T=("person_mean_T", "median"),
              median_sun=("person_mean_sun", "median"),
              median_wind=("person_mean_wind", "median"),
          )
    )

    print(f"\n=== Person-weighted averages (each {person_id_col} = 1) ===")
    print(person_weighted)

    if set(person_weighted[gender_col]) >= {"Woman", "Man"}:
        pw = person_weighted.set_index(gender_col)
        print("\nPerson-weighted Woman - Man:")
        print(pd.Series({
            "ΔT (°C)": pw.loc["Woman", "mean_T"] - pw.loc["Man", "mean_T"],
            "Δsun index": pw.loc["Woman", "mean_sun"] - pw.loc["Man", "mean_sun"],
            "Δwind index": pw.loc["Woman", "mean_wind"] - pw.loc["Man", "mean_wind"],
        }))
else:
    print(
        "\nNOTE: No obvious person-id column found, so I can’t do true person-by-person weighting.\n"
        "If you tell me the participant id column name, I’ll plug it in."
    )

In [ ]:
import numpy as np
import pandas as pd

temp_col = "<T-T_fixed+<T>>"
gender_col = "gender"
tsv_col = "thermal_sensation"
sun_col = "sun"
wind_col = "wind"

wind_map = {
    "A strong wind": -1,
    "A moderate wind": -1,
    "A light wind": 0,
    "A very light wind": 0,
    "It's not windy": 1,
}

sun_map = {
    "In full shade": -1,
    "In a mixture of sun and shadow": 0,
    "In full sun": 1,
}

# --- parameters ---
low_quantile = 0.25   # bottom 25% temperatures within (Woman, Very hot)
# low_quantile = 0.20 # e.g. bottom 20%
# low_quantile = 0.33 # e.g. bottom third

df = df_votes.copy()
df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")
df["sun_num"] = df[sun_col].map(sun_map)
df["wind_num"] = df[wind_col].map(wind_map)
df["walk"] = df["space_code"].astype(str).str[:2]

# 1) Women who voted Very hot
vh_w = df[
    (df[gender_col] == "Woman") &
    (df[tsv_col] == "Very hot")
].dropna(subset=[temp_col, sun_col, wind_col, "sun_num", "wind_num"]).copy()

print("n (Woman & Very hot):", len(vh_w))
if vh_w.empty:
    raise SystemExit("No rows found for Woman & Very hot after dropping NAs.")

# 2) Define "lower temperature" subgroup
thr = vh_w[temp_col].quantile(low_quantile)
vh_w["low_T_group"] = np.where(vh_w[temp_col] <= thr, "Low-T women (Very hot)", "Other women (Very hot)")

print(f"Low-T threshold (q={low_quantile:.2f}): T <= {thr:.3f} °C")

# 3) Numeric summaries
num_summary = (
    vh_w.groupby("low_T_group", as_index=False)
        .agg(
            n=(temp_col, "count"),
            mean_T=(temp_col, "mean"),
            std_T=(temp_col, "std"),
            mean_sun=("sun_num", "mean"),
            mean_wind=("wind_num", "mean"),
        )
)
print("\n=== Numeric summary ===")
print(num_summary)

# 4) Sun/wind category distributions (counts + %)
def cat_dist(df_in: pd.DataFrame, col: str) -> pd.DataFrame:
    c = df_in[col].value_counts(dropna=False)
    p = df_in[col].value_counts(normalize=True, dropna=False) * 100
    out = pd.DataFrame({"count": c, "pct": p.round(2)})
    return out

for grp, gdf in vh_w.groupby("low_T_group"):
    print(f"\n=== {grp}: SUN distribution ===")
    print(cat_dist(gdf, sun_col))
    print(f"\n=== {grp}: WIND distribution ===")
    print(cat_dist(gdf, wind_col))

# 5) Optional: where are these low-T very-hot votes happening? (top walks)
low_only = vh_w[vh_w["low_T_group"] == "Low-T women (Very hot)"]
print("\n=== Top walks for Low-T women (Very hot) ===")
print(low_only["walk"].value_counts().head(10))

## sembla que la temperatura era baixsa però res de l'altre mon respecte al sol i això. simplement suposo k tenien calor. Estaria bé mirar-ho amb el comfort que tenien

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

temp_col = "<T-T_fixed+<T>>"
y_col = "thermal_sensation"
gender_col = "gender"

tsv_map = {
    "Cool": 1,
    "Slightly cool": 2,
    "Neutral": 3,
    "Slightly warm": 4,
    "Warm": 5,
    "Hot": 6,
    "Very hot": 7
}

sun_map = {
    "In full shade": -1,
    "In a mixture of sun and shadow": 0,
    "In full sun": 1,
}

wind_map = {
    "A strong wind": -1,
    "A moderate wind": -1,
    "A light wind": 0,
    "A very light wind": 0,
    "It's not windy": 1,
}

df = df_votes.copy()
df["walk"] = df["space_code"].astype(str).str[:2]
df["T"] = pd.to_numeric(df[temp_col], errors="coerce")
df["TSV"] = df[y_col].map(tsv_map)
df["sun_num"] = df["sun"].map(sun_map)
df["wind_num"] = df["wind"].map(wind_map)

# keep Woman/Man (edit if needed)
df = df[df[gender_col].isin(["Woman", "Man"])].copy()

df = df.dropna(subset=["walk", "T", "TSV", "sun_num", "wind_num", gender_col]).copy()

# -------------------------------------------------------
# 1) Remove sun/wind + walk effects from TSV
# -------------------------------------------------------
adj_model = smf.ols("TSV ~ sun_num + wind_num + C(walk)", data=df).fit()

# residual TSV: what's left after removing sun/wind/walk
# add back overall mean so it's still on a "TSV scale"
df["TSV_adj"] = adj_model.resid + df["TSV"].mean()

print(adj_model.summary())

# -------------------------------------------------------
# 2) Bin temperature (shared bins) and plot mean TSV_adj by gender
# -------------------------------------------------------
n_bins = 12
edges = np.linspace(df["T"].min(), df["T"].max(), n_bins + 1)
centers = (edges[:-1] + edges[1:]) / 2

df["T_bin"] = pd.cut(df["T"], edges, include_lowest=True)
center_map = dict(zip(df["T_bin"].cat.categories, centers))

def summarize(sub_df: pd.DataFrame) -> pd.DataFrame:
    s = (
        sub_df.groupby("T_bin", observed=False)
        .agg(mean_y=("TSV_adj", "mean"), std_y=("TSV_adj", "std"), n=("TSV_adj", "count"))
        .dropna()
        .reset_index()
    )
    s["sem_y"] = s["std_y"] / np.sqrt(s["n"])
    s["x"] = s["T_bin"].map(center_map)
    return s.sort_values("x")

plt.figure(figsize=(8, 5))
for g in ["Woman", "Man"]:
    sub = df[df[gender_col] == g]
    if sub.empty:
        continue
    s = summarize(sub)
    plt.errorbar(s["x"], s["mean_y"], yerr=s["sem_y"], fmt="o-", capsize=4, label=g)

plt.xlabel("Corrected Temperature (°C)")
plt.ylabel("Adjusted TSV (sun/wind/walk removed)")
plt.title("Gender curves vs temperature (TSV adjusted for sun, wind, and walk)")
plt.yticks([1, 2, 3, 4, 5, 6, 7])
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# A PARTIR D'ARA VENEN UN MUNT DE CÀLCULS SOBRE SI SOBREN PER L'ANÀLISI LES CATEGORIES EXTREMALS

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

temp_col = "<T-T_fixed+<T>>"
gender_col = "gender"
sens_col = "thermal_sensation"

sun_map = {"In full shade":-1, "In a mixture of sun and shadow":0, "In full sun":1}
wind_map = {"A strong wind":-1, "A moderate wind":-1, "A light wind":0, "A very light wind":0, "It's not windy":1}

df = df_votes.copy()
df["T"] = pd.to_numeric(df[temp_col], errors="coerce")
df["sun_num"] = df["sun"].map(sun_map)
df["wind_num"] = df["wind"].map(wind_map)
df = df.dropna(subset=["T","sun_num","wind_num",gender_col,sens_col]).copy()
df = df[df[gender_col].isin(["Woman","Man"])].copy()

# heat score (simple; you can tune weights)
df["heat_score"] = 0.5*df["T"] + 2*df["sun_num"] + 1*df["wind_num"]

def auc_for_gender(g):
    sub = df[df[gender_col]==g].copy()
    y = (sub[sens_col] == "Very hot").astype(int).to_numpy()
    x = sub["heat_score"].to_numpy()
    if y.sum() < 5 or (y==0).sum() < 5:
        return np.nan
    return roc_auc_score(y, x)

print("AUC (Very hot vs others) using heat_score:")
print("  Woman:", auc_for_gender("Woman"))
print("  Man:  ", auc_for_gender("Man"))

In [ ]:
import numpy as np
import pandas as pd

space_col = "space_code"
gender_col = "gender"
sens_col = "thermal_sensation"

df = df_votes.copy()

# stop_id = first 2 chars (town+walk) + trailing stop digits
df["walk_prefix"] = df[space_col].astype(str).str[:2]
df["stop_num"] = pd.to_numeric(df[space_col].astype(str).str.extract(r"(\d+)\s*$")[0], errors="coerce")
df["stop_id"] = df["walk_prefix"] + "_" + df["stop_num"].astype("Int64").astype(str)

df = df.dropna(subset=["stop_id", gender_col, sens_col]).copy()
df = df[df[gender_col].isin(["Woman", "Man"])].copy()

def stop_level_corr(label: str):
    sub = df.copy()
    sub["is_label"] = (sub[sens_col] == label).astype(int)

    # per stop + gender: fraction of this label
    stop_gender = (
        sub.groupby(["stop_id", gender_col], as_index=False)
           .agg(n=("is_label", "count"), frac=("is_label", "mean"))
    )

    wide = stop_gender.pivot(index="stop_id", columns=gender_col, values="frac").dropna()

    # also keep how much data each stop has for each gender (for filtering)
    n_wide = stop_gender.pivot(index="stop_id", columns=gender_col, values="n").dropna()

    corr = wide["Woman"].corr(wide["Man"])

    return corr, wide, n_wide

# Correlations
corr_vh, wide_vh, n_vh = stop_level_corr("Very hot")
corr_cool, wide_cool, n_cool = stop_level_corr("Cool")

print("Correlation across stops (Women vs Men fraction):")
print("  Very hot:", corr_vh)
print("  Cool:    ", corr_cool)

# OPTIONAL: filter out stops with too few votes per gender (stabilizes correlation)
min_n = 5
mask_vh = (n_vh["Woman"] >= min_n) & (n_vh["Man"] >= min_n)
mask_c = (n_cool["Woman"] >= min_n) & (n_cool["Man"] >= min_n)

print(f"\nFiltered correlations (only stops with >= {min_n} votes for EACH gender):")
print("  Very hot:", wide_vh.loc[mask_vh, "Woman"].corr(wide_vh.loc[mask_vh, "Man"]))
print("  Cool:    ", wide_cool.loc[mask_c, "Woman"].corr(wide_cool.loc[mask_c, "Man"]))

In [ ]:

space_col = "space_code"
gender_col = "gender"
sens_col = "thermal_sensation"

df = df_votes.copy()
df["walk_prefix"] = df[space_col].astype(str).str[:2]
df["stop_num"] = pd.to_numeric(df[space_col].astype(str).str.extract(r"(\d+)\s*$")[0], errors="coerce")
df["stop_id"] = df["walk_prefix"] + "_" + df["stop_num"].astype("Int64").astype(str)

df = df.dropna(subset=["stop_id", gender_col, sens_col]).copy()
df = df[df[gender_col].isin(["Woman", "Man"])].copy()

def split_half_reliability(label: str, gender: str, min_n: int = 8, seed: int = 42) -> float:
    rng = np.random.default_rng(seed)
    sub = df[df[gender_col] == gender].copy()
    sub["is_label"] = (sub[sens_col] == label).astype(int)

    fracs_a = {}
    fracs_b = {}

    for stop_id, g in sub.groupby("stop_id"):
        n = len(g)
        if n < min_n:
            continue
        idx = np.arange(n)
        rng.shuffle(idx)
        a = g.iloc[idx[: n // 2]]
        b = g.iloc[idx[n // 2 :]]
        fracs_a[stop_id] = a["is_label"].mean()
        fracs_b[stop_id] = b["is_label"].mean()

    if len(fracs_a) < 10:
        return np.nan  # not enough stops to estimate reliably

    s1 = pd.Series(fracs_a)
    s2 = pd.Series(fracs_b).reindex(s1.index)
    return float(s1.corr(s2))

for label in ["Very hot", "Cool"]:
    for gender in ["Woman", "Man"]:
        r = split_half_reliability(label, gender, min_n=6, seed=42)
        print(f"Split-half r | label={label:8s} gender={gender:5s}: {r}")

In [ ]:
cool_side_man_votes = df_votes[
    (df_votes["gender"] == "Man") &
    (df_votes["thermal_sensation"].isin(["Cool"]))
].shape[0]
print("Cool-side (Man) votes:", cool_side_man_votes)

In [ ]:
import numpy as np
import pandas as pd

space_col = "space_code"
gender_col = "gender"
sens_col = "thermal_sensation"

df = df_votes.copy()
df["walk_prefix"] = df[space_col].astype(str).str[:2]
df["stop_num"] = pd.to_numeric(df[space_col].astype(str).str.extract(r"(\d+)\s*$")[0], errors="coerce")
df["stop_id"] = df["walk_prefix"] + "_" + df["stop_num"].astype("Int64").astype(str)

df = df.dropna(subset=["stop_id", gender_col, sens_col]).copy()
df = df[df[gender_col].isin(["Woman", "Man"])].copy()

df["is_cool_side"] = df[sens_col].isin(["Cool", "Slightly cool"]).astype(int)

# Between-gender corr across stops
stop_gender = (
    df.groupby(["stop_id", gender_col], as_index=False)
      .agg(n=("is_cool_side","count"), frac=("is_cool_side","mean"))
)
wide = stop_gender.pivot(index="stop_id", columns=gender_col, values="frac").dropna()
nwide = stop_gender.pivot(index="stop_id", columns=gender_col, values="n").dropna()

print("Cool-side corr (raw):", wide["Woman"].corr(wide["Man"]))

min_n = 4
mask = (nwide["Woman"]>=min_n) & (nwide["Man"]>=min_n)
print(f"Cool-side corr (>= {min_n} votes each):", wide.loc[mask,"Woman"].corr(wide.loc[mask,"Man"]))

# Within-gender split-half reliability (one random split)
def split_half_r(label_col: str, gender: str, min_n: int = 4, seed: int = 42):
    rng = np.random.default_rng(seed)
    sub = df[df[gender_col] == gender].copy()

    fracs_a, fracs_b = {}, {}
    for stop_id, g in sub.groupby("stop_id"):
        n = len(g)
        if n < min_n:
            continue
        idx = np.arange(n)
        rng.shuffle(idx)
        a = g.iloc[idx[: n // 2]]
        b = g.iloc[idx[n // 2 :]]
        fracs_a[stop_id] = a[label_col].mean()
        fracs_b[stop_id] = b[label_col].mean()

    if len(fracs_a) < 5:
        return np.nan
    s1 = pd.Series(fracs_a)
    s2 = pd.Series(fracs_b).reindex(s1.index)
    return float(s1.corr(s2))

print("Split-half r (Cool-side) Woman:", split_half_r("is_cool_side", "Woman", min_n=4))
print("Split-half r (Cool-side) Man:  ", split_half_r("is_cool_side", "Man", min_n=4))

In [ ]:
import numpy as np
import pandas as pd

space_col = "space_code"
gender_col = "gender"
sens_col = "thermal_sensation"

df = df_votes.copy()
df["walk_prefix"] = df[space_col].astype(str).str[:2]
df["stop_num"] = pd.to_numeric(df[space_col].astype(str).str.extract(r"(\d+)\s*$")[0], errors="coerce")
df["stop_id"] = df["walk_prefix"] + "_" + df["stop_num"].astype("Int64").astype(str)

df = df.dropna(subset=["stop_id", gender_col, sens_col]).copy()
df = df[df[gender_col].isin(["Woman", "Man"])].copy()

df["is_veryhot"] = (df[sens_col] == "Very hot").astype(int)
df["is_hotside"] = df[sens_col].isin(["Hot", "Very hot"]).astype(int)

def between_gender_corr(label_col: str, min_n: int | None = None):
    stop_gender = (
        df.groupby(["stop_id", gender_col], as_index=False)
          .agg(n=(label_col, "count"), frac=(label_col, "mean"))
    )
    wide = stop_gender.pivot(index="stop_id", columns=gender_col, values="frac").dropna()
    if min_n is None:
        return float(wide["Woman"].corr(wide["Man"]))

    nwide = stop_gender.pivot(index="stop_id", columns=gender_col, values="n").dropna()
    mask = (nwide["Woman"] >= min_n) & (nwide["Man"] >= min_n)
    return float(wide.loc[mask, "Woman"].corr(wide.loc[mask, "Man"]))

def split_half_r(label_col: str, gender: str, min_n: int = 4, seed: int = 42):
    rng = np.random.default_rng(seed)
    sub = df[df[gender_col] == gender].copy()

    fracs_a, fracs_b = {}, {}
    for stop_id, g in sub.groupby("stop_id"):
        n = len(g)
        if n < min_n:
            continue
        idx = np.arange(n)
        rng.shuffle(idx)
        a = g.iloc[idx[: n // 2]]
        b = g.iloc[idx[n // 2 :]]
        fracs_a[stop_id] = a[label_col].mean()
        fracs_b[stop_id] = b[label_col].mean()

    if len(fracs_a) < 5:
        return np.nan
    s1 = pd.Series(fracs_a)
    s2 = pd.Series(fracs_b).reindex(s1.index)
    return float(s1.corr(s2))

min_n = 4

for name, col in [("Very hot", "is_veryhot"), ("Hot-side (Hot+Very hot)", "is_hotside")]:
    print(f"\n=== {name} ===")
    print("Between-gender corr (raw):", between_gender_corr(col, min_n=None))
    print(f"Between-gender corr (>= {min_n} votes each):", between_gender_corr(col, min_n=min_n))
    print(f"Split-half r Woman (min_n={min_n}):", split_half_r(col, "Woman", min_n=min_n))
    print(f"Split-half r Man   (min_n={min_n}):", split_half_r(col, "Man", min_n=min_n))

In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.metrics import roc_auc_score

temp_col = "<T-T_fixed+<T>>"
sens_col = "thermal_sensation"

sun_map = {"In full shade":-1, "In a mixture of sun and shadow":0, "In full sun":1}
wind_map = {"A strong wind":-1, "A moderate wind":-1, "A light wind":0, "A very light wind":0, "It's not windy":1}

cats = ["Cool","Slightly cool","Neutral","Slightly warm","Warm","Hot","Very hot"]

df = df_votes.copy()
df["T"] = pd.to_numeric(df[temp_col], errors="coerce")
df["sun_num"] = df["sun"].map(sun_map)
df["wind_num"] = df["wind"].map(wind_map)
df = df.dropna(subset=["T", sens_col]).copy()
df = df[df[sens_col].isin(cats)].copy()

# choose score: temperature only OR heat_score
df["score_T"] = df["T"]
df["score_heat"] = df["T"] + 0.7*df["sun_num"].fillna(0) + 0.7*df["wind_num"].fillna(0)

def pairwise_auc(score_col: str):
    rows = []
    for a, b in combinations(cats, 2):
        sub = df[df[sens_col].isin([a,b])].copy()
        if sub.shape[0] < 30:
            continue
        y = (sub[sens_col] == b).astype(int).to_numpy()
        x = sub[score_col].to_numpy()
        auc = roc_auc_score(y, x)
        rows.append({"cat_a": a, "cat_b": b, "auc": auc, "n": len(sub)})
    out = pd.DataFrame(rows).sort_values("auc")
    return out

print("=== Most-overlapping pairs b y Temperature only (AUC near 0.5) ===")
print(pairwise_auc("score_T").head(15))

print("\n=== Most-overlapping pairs by Heat score (AUC near 0.5) ===")
print(pairwise_auc("score_heat").head(15))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from sklearn.metrics import roc_auc_score

temp_col = "<T-T_fixed+<T>>"
sens_col = "thermal_sensation"

sun_map = {"In full shade":-1, "In a mixture of sun and shadow":0, "In full sun":1}
wind_map = {"A strong wind":-1, "A moderate wind":-1, "A light wind":0, "A very light wind":0, "It's not windy":1}

cats = ["Cool","Slightly cool","Neutral","Slightly warm","Warm","Hot","Very hot"]

df = df_votes.copy()
df["T"] = pd.to_numeric(df[temp_col], errors="coerce")
df["sun_num"] = df["sun"].map(sun_map)
df["wind_num"] = df["wind"].map(wind_map)

df = df.dropna(subset=["T", sens_col]).copy()
df = df[df[sens_col].isin(cats)].copy()

# choose score:
score_col = "score_T"  # change to "score_T" for temperature-only
df["score_T"] = df["T"]
df["score_heat"] = df["T"] + 0.7*df["sun_num"].fillna(0) + 0.7*df["wind_num"].fillna(0)

# matrices
auc_mat = pd.DataFrame(np.nan, index=cats, columns=cats, dtype=float)
overlap_mat = pd.DataFrame(np.nan, index=cats, columns=cats, dtype=float)

for a, b in combinations(cats, 2):
    sub = df[df[sens_col].isin([a, b])].copy()
    if len(sub) < 30:
        continue
    y = (sub[sens_col] == b).astype(int).to_numpy()
    x = sub[score_col].to_numpy()

    auc = roc_auc_score(y, x)
    # fill both directions
    auc_mat.loc[a, b] = auc
    auc_mat.loc[b, a] = 1 - auc

# convert to symmetric overlap in [0,1]
# overlap = 1 - 2*abs(AUC - 0.5)
overlap_mat = 1 - 2 * (auc_mat - 0.5).abs()
np.fill_diagonal(overlap_mat.values, 1.0)

print("=== AUC matrix (directional) ===")
print(auc_mat.round(3))

print("\n=== Overlap matrix (symmetric; 1=lots of overlap, 0=separable) ===")
print(overlap_mat.round(3))

# plot overlap heatmap (matplotlib only)
plt.figure(figsize=(9, 7))
plt.imshow(overlap_mat.values, aspect="auto")
plt.xticks(range(len(cats)), cats, rotation=45, ha="right")
plt.yticks(range(len(cats)), cats)
plt.colorbar(label="Overlap (1 = indistinguishable, 0 = well separated)")
plt.title(f"Category overlap map using {score_col}")
plt.tight_layout()
 #plt.show()

## Now we try to separate "COLD/WARM" walks, to see if there are visible differences

In [ ]:

# ----------------------------
temp_col = "<T-T_fixed+<T>>"
y_col = "thermal_sensation"

tsv_map = {
    "Slightly cool": 1,
    "Cool": 2,
    "Neutral": 3,
    "Slightly warm": 4,
    "Warm": 5,
    "Hot": 6,
    "Very hot": 7,
}
y_ticks = list(range(1, 8))
y_labels = [k for k, v in sorted(tsv_map.items(), key=lambda kv: kv[1])]

# ----------------------------
# Prepare dataframe
# ----------------------------
df = df_votes.copy()
df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")
df["walk"] = df["space_code"].astype(str).str[:2]
df["TSV"] = df[y_col].map(tsv_map)

df = df.dropna(subset=[temp_col, "walk", "TSV"]).copy()
df["TSV"] = df["TSV"].astype(int)

# Walk mean temperature (°C) and centered temperature (°C)
df["T_mean_walk_C"] = df.groupby("walk")[temp_col].transform("mean")
df["T_centered_walk_C"] = df[temp_col] - df["T_mean_walk_C"]

# ----------------------------
# Walk-level summary + Cold/Warm split (UNWEIGHTED by votes)
# ----------------------------
walk_summary = (
    df.groupby("walk", as_index=False)["T_mean_walk_C"]
    .first()
    .rename(columns={"T_mean_walk_C": "walk_mean_C"})
)

# 2 quantiles => Cold vs Warm (roughly equal number of walks)
walk_summary["walk_group"] = pd.qcut(
    walk_summary["walk_mean_C"],
    q=2,
    labels=["Cold walks", "Warm walks"],
)

# Map group back to rows
df["walk_group"] = df["walk"].map(walk_summary.set_index("walk")["walk_group"])

# Average temperature of each group = mean of walk means (NOT vote-weighted)
group_avg = (
    walk_summary.groupby("walk_group", as_index=False)["walk_mean_C"]
    .mean()
    .rename(columns={"walk_mean_C": "avg_walk_mean_C"})
)
print("=== Average temperature by group (mean of walk means, °C) ===")
print(group_avg)

avg_map = group_avg.set_index("walk_group")["avg_walk_mean_C"].to_dict()

# ----------------------------
# Summarize: mean centered T for each TSV (within each walk_group)
# ----------------------------
def summarize_mean_T_for_TSV(sub_df: pd.DataFrame) -> pd.DataFrame:
    s = (
        sub_df.groupby("TSV", as_index=False)["T_centered_walk_C"]
        .agg(mean_T="mean", std_T="std", n="count")
        .dropna()
        .sort_values("TSV")
    )
    s["sem_T"] = s["std_T"] / np.sqrt(s["n"])
    return s

# ----------------------------
# Plot: mean T_centered vs TSV (Cold vs Warm walks)
# ----------------------------
plt.figure(figsize=(8, 5))

for grp in ["Cold walks", "Warm walks"]:
    sub = df[df["walk_group"] == grp].dropna(subset=["TSV", "T_centered_walk_C"])
    if sub.empty:
        continue

    summary = summarize_mean_T_for_TSV(sub)
    if summary.empty:
        continue

    avg_walk_mean = avg_map.get(grp, float("nan"))

    plt.errorbar(
        summary["mean_T"],     # x = mean centered temperature
        summary["TSV"],        # y = TSV category
        xerr=summary["sem_T"], # horizontal SEM on temperature
        fmt="o-",
        capsize=4,
        label=f"{grp} (avg walk mean = {avg_walk_mean:.2f}°C)",
    )

plt.yticks(y_ticks, y_labels)
plt.ylim(0.5, 7.5)
plt.axvline(0, linewidth=1, alpha=0.5)  # 0 = walk mean
plt.xlabel("Mean temperature centered by walk mean (°C)")
plt.ylabel("Thermal Sensation")
plt.title("Mean centered temperature for each TSV, split by cold vs warm walks")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Sembla que el pendent és el mateix apx i que no hi ha massa diferències

# Ara anem a mirar mean TSV x T bin

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# Base columns
# ----------------------------
temp_col = "<T-T_fixed+<T>>"
y_col = "thermal_sensation"

tsv_map = {
    "Slightly cool": 1,
    "Cool": 2,
    "Neutral": 3,
    "Slightly warm": 4,
    "Warm": 5,
    "Hot": 6,
    "Very hot": 7,
}

# ----------------------------
# Prepare dataframe (no walk split)
# ----------------------------
df = df_votes.copy()

df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")
df["TSV"] = df[y_col].map(tsv_map)
df = df.dropna(subset=[temp_col, "TSV"]).copy()

# ----------------------------
# Make temperature bins
# ----------------------------
n_bins = 10  # change if you want
tmin = df[temp_col].min()
tmax = df[temp_col].max()
bin_edges = np.linspace(tmin, tmax, n_bins + 1)

df["T_bin"] = pd.cut(df[temp_col], bins=bin_edges, include_lowest=True)

# ----------------------------
# Mean TSV per bin (+ SEM)
# ----------------------------
summary = (
    df.groupby("T_bin", observed=False)
    .agg(
        mean_T=(temp_col, "mean"),
        mean_TSV=("TSV", "mean"),
        std_TSV=("TSV", "std"),
        n=("TSV", "count"),
    )
    .dropna()
    .reset_index()
    .sort_values("mean_T")
)
summary["sem_TSV"] = summary["std_TSV"] / np.sqrt(summary["n"])

print(summary[["T_bin", "mean_T", "mean_TSV", "n", "sem_TSV"]])

# ----------------------------
# Plot: mean TSV vs temperature bin mean
# ----------------------------
plt.figure(figsize=(8, 5))
plt.errorbar(
    summary["mean_T"],
    summary["mean_TSV"],
    yerr=summary["sem_TSV"],
    fmt="o-",
    capsize=4,
)

plt.xlabel("Temperature (°C) (bin mean)")
plt.ylabel("Mean TSV")
plt.title("Mean TSV by temperature bins")
plt.yticks([1, 2, 3, 4, 5, 6, 7])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## En 3 grups

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# Base columns
# ----------------------------
temp_col = "<T-T_fixed+<T>>"
y_col = "thermal_sensation"

tsv_map = {
    "Slightly cool": 1,
    "Cool": 1,
    "Neutral": 2,
    "Slightly warm": 2,
    "Warm": 3,
    "Hot": 3,
    "Very hot": 3,
}

# ----------------------------
# Prepare dataframe (no walk split)
# ----------------------------
df = df_votes.copy()
df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")
df["TSV"] = df[y_col].map(tsv_map)
df = df.dropna(subset=[temp_col, "TSV"]).copy()

# ----------------------------
# Make temperature bins
# ----------------------------
n_bins = 10  # change if you want
tmin = df[temp_col].min()
tmax = df[temp_col].max()
bin_edges = np.linspace(tmin, tmax, n_bins + 1)

df["T_bin"] = pd.cut(df[temp_col], bins=bin_edges, include_lowest=True)

# ----------------------------
# Mean TSV per bin (+ SEM)
# ----------------------------
summary = (
    df.groupby("T_bin", observed=False)
    .agg(
        mean_T=(temp_col, "mean"),
        mean_TSV=("TSV", "mean"),
        std_TSV=("TSV", "std"),
        n=("TSV", "count"),
    )
    .dropna()
    .reset_index()
    .sort_values("mean_T")
)
summary["sem_TSV"] = summary["std_TSV"] / np.sqrt(summary["n"])

print(summary[["T_bin", "mean_T", "mean_TSV", "n", "sem_TSV"]])

# ----------------------------
# Plot: mean TSV vs temperature bin mean
# ----------------------------
plt.figure(figsize=(8, 5))
plt.errorbar(
    summary["mean_T"],
    summary["mean_TSV"],
    yerr=summary["sem_TSV"],
    fmt="o-",
    capsize=4,
)

plt.xlabel("Temperature (°C) (bin mean)")
plt.ylabel("Mean TSV")
plt.title("Mean TSV by temperature bins")
plt.yticks([1, 2, 3])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# Base columns
# ----------------------------
temp_col = "<T-T_fixed+<T>>"      # corrected temperature column
y_col = "thermal_sensation"
gender_col = "gender"

tsv_map = {
    "Slightly cool": 1,
    "Cool": 2,
    "Neutral": 3,
    "Slightly warm": 4,
    "Warm": 5,
    "Hot": 6,
    "Very hot": 7,
}

# ----------------------------
# Prepare dataframe (no walk-centering)
# ----------------------------
df = df_votes.copy()
df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")
df["TSV"] = df[y_col].map(tsv_map)

df = df.dropna(subset=[temp_col, "TSV", gender_col]).copy()
df["TSV"] = df["TSV"].astype(int)

# ----------------------------
# Shared temperature bins (global) + bin centers
# ----------------------------
n_bins = 12  # tweak if you want
tmin = df[temp_col].min()
tmax = df[temp_col].max()
bin_edges = np.linspace(tmin, tmax, n_bins + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

df["T_bin"] = pd.cut(df[temp_col], bins=bin_edges, include_lowest=True)
cats = df["T_bin"].cat.categories
center_map = dict(zip(cats, bin_centers))

def summarize_mean_tsv_by_bin(sub_df: pd.DataFrame) -> pd.DataFrame:
    s = (
        sub_df.groupby("T_bin", observed=False)
        .agg(
            mean_TSV=("TSV", "mean"),
            std_TSV=("TSV", "std"),
            n=("TSV", "count"),
        )
        .dropna()
        .reset_index()
    )
    s["sem_TSV"] = s["std_TSV"] / np.sqrt(s["n"])
    s["x_center"] = s["T_bin"].map(center_map)
    return s.sort_values("x_center")

# ----------------------------
# Plot by gender: mean TSV vs corrected temperature
# ----------------------------
plt.figure(figsize=(8, 5))

for gender in ["Woman", "Man"]:
    sub = df[df[gender_col] == gender].dropna(subset=["T_bin", "TSV"])
    if sub.empty:
        continue

    summary = summarize_mean_tsv_by_bin(sub)
    if summary.empty:
        continue

    plt.errorbar(
        summary["x_center"],
        summary["mean_TSV"],
        yerr=summary["sem_TSV"],
        fmt="o-",
        capsize=4,
        label=gender,
    )

plt.xlabel("Corrected Temperature (°C)")
plt.ylabel("Mean TSV")
plt.title("Mean TSV vs corrected temperature, split by gender")
plt.yticks([1, 2, 3, 4, 5, 6, 7])
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## i ara anem a diferenciar per COLD/WARM caminades

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# Base columns
# ----------------------------
temp_col = "<T-T_fixed+<T>>"
y_col = "thermal_sensation"

tsv_map = {
    "Slightly cool": 1,
    "Cool": 2,
    "Neutral": 3,
    "Slightly warm": 4,
    "Warm": 5,
    "Hot": 6,
    "Very hot": 7,
}

# ----------------------------
# Prepare dataframe
# ----------------------------
df = df_votes.copy()
df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")
df["walk"] = df["space_code"].astype(str).str[:2]
df["TSV"] = df[y_col].map(tsv_map)

df = df.dropna(subset=[temp_col, "walk", "TSV"]).copy()
df["TSV"] = df["TSV"].astype(int)

# Walk mean temperature (°C) and centered temp (°C)
df["T_mean_walk_C"] = df.groupby("walk")[temp_col].transform("mean")
df["T_centered_walk_C"] = df[temp_col] - df["T_mean_walk_C"]

# ----------------------------
# Walk-level table + Cold/Warm split
# ----------------------------
walk_summary = (
    df.groupby("walk", as_index=False)["T_mean_walk_C"]
    .first()
    .rename(columns={"T_mean_walk_C": "walk_mean_C"})
)

# 2 quantiles => Cold vs Warm (roughly equal number of walks)
walk_summary["walk_group"] = pd.qcut(
    walk_summary["walk_mean_C"],
    q=2,
    labels=["Cold walks", "Warm walks"],
)

df["walk_group"] = df["walk"].map(walk_summary.set_index("walk")["walk_group"])

# ----------------------------
# Print average temperature per group
# (average of walk means; not weighted by #votes)
# ----------------------------
group_avg = walk_summary.groupby("walk_group", as_index=False)["walk_mean_C"].agg(
    avg_walk_mean_C="mean",
    n_walks="count",
)
print("=== Average temperature by group (mean of walk means) ===")
print(group_avg)

# (Optional) also vote-weighted mean temperature, if you want
vote_weighted = (
    df.groupby("walk_group", as_index=False)[temp_col]
    .agg(avg_vote_weighted_T_C="mean", n_votes="count")
)
print("\n=== Vote-weighted average temperature by group ===")
print(vote_weighted)

# ----------------------------
# Shared bins for centered temperature (so curves are comparable)
# ----------------------------
n_bins = 10
tmin = df["T_centered_walk_C"].min()
tmax = df["T_centered_walk_C"].max()
bin_edges = np.linspace(tmin, tmax, n_bins + 1)
df["T_bin"] = pd.cut(df["T_centered_walk_C"], bins=bin_edges, include_lowest=True)

def summarize_mean_tsv_by_bin(sub_df: pd.DataFrame) -> pd.DataFrame:
    s = (
        sub_df.groupby("T_bin", observed=False)
        .agg(
            mean_T=("T_centered_walk_C", "mean"),
            mean_TSV=("TSV", "mean"),
            std_TSV=("TSV", "std"),
            n=("TSV", "count"),
        )
        .dropna()
        .reset_index()
        .sort_values("mean_T")
    )
    s["sem_TSV"] = s["std_TSV"] / np.sqrt(s["n"])
    return s

# ----------------------------
# Plot: Mean TSV vs centered temperature, Cold vs Warm walks
# ----------------------------
plt.figure(figsize=(8, 5))

for grp in ["Cold walks", "Warm walks"]:
    sub = df[df["walk_group"] == grp].dropna(subset=["T_bin", "TSV", "T_centered_walk_C"])
    if sub.empty:
        continue

    summary = summarize_mean_tsv_by_bin(sub)
    if summary.empty:
        continue

    plt.errorbar(
        summary["mean_T"],
        summary["mean_TSV"],
        yerr=summary["sem_TSV"],
        fmt="o-",
        capsize=4,
        label=f"{grp} (n={int(summary['n'].sum())})",
    )

plt.xlabel("Temperature centered by walk mean (°C)")
plt.ylabel("Mean TSV")
plt.title("Mean TSV vs centered temperature (by-walk), split by cold vs warm walks")
plt.yticks([1, 2, 3, 4, 5, 6, 7])
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

### Sembla que simplement tenim un desplaçament vertical i ben bé k prou més. Els valors no estàn en el mateix x per a tenir prous valors per fer l'average (cold walks i warm walks no tenen les mateixes dades)

## Ara per gènere i per edat

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# Base columns
# ----------------------------
temp_col = "<T-T_fixed+<T>>"
y_col = "thermal_sensation"
gender_col = "gender"
age_col = "age2"

tsv_map = {
    "Slightly cool": 1,
    "Cool": 2,
    "Neutral": 3,
    "Slightly warm": 4,
    "Warm": 5,
    "Hot": 6,
    "Very hot": 7
}

# ----------------------------
# Prepare dataframe
# ----------------------------
df_votes["walk"] = df_votes["space_code"].astype(str).str[:2]
df_votes["T_K"] = df_votes[temp_col] + 273.15
df_votes["T_mean_walk_K"] = df_votes.groupby("walk")["T_K"].transform("mean")
df_votes["T_centered_walk_K"] = df_votes["T_K"] - df_votes["T_mean_walk_K"]
df_votes["TSV"] = df_votes[y_col].map(tsv_map)

# ----------------------------
# Classify walks as Cold / Medium / Hot
# ----------------------------
walk_summary = (
    df_votes.groupby("walk", as_index=False)["T_mean_walk_K"]
    .first()
    .rename(columns={"T_mean_walk_K": "walk_mean_K"})
)

walk_summary["walk_group"] = pd.qcut(
    walk_summary["walk_mean_K"],
    q=3,
    labels=["Cold walks", "Medium walks", "Hot walks"]
)

df_votes["walk_group"] = df_votes["walk"].map(
    walk_summary.set_index("walk")["walk_group"]
)

# =========================================================
# 1) PLOT BY GENDER
# =========================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, walk_group in zip(axes, ["Cold walks", "Medium walks", "Hot walks"]):
    group_df = df_votes[df_votes["walk_group"] == walk_group].copy()

    for gender in ["Woman", "Man"]:
        gdf = group_df[group_df[gender_col] == gender].copy()
        gdf = gdf[["T_centered_walk_K", "TSV"]].dropna()

        if len(gdf) == 0:
            continue

        # bin temperature
        gdf["T_bin"] = pd.cut(gdf["T_centered_walk_K"], bins=8)

        summary = (
            gdf.groupby("T_bin", observed=False)
            .agg(
                mean_T=("T_centered_walk_K", "mean"),
                mean_TSV=("TSV", "mean"),
                std_TSV=("TSV", "std"),
                n=("TSV", "count")
            )
            .dropna()
            .reset_index()
            .sort_values("mean_T")
        )

        if len(summary) == 0:
            continue

        summary["sem_TSV"] = summary["std_TSV"] / np.sqrt(summary["n"])

        ax.errorbar(
            summary["mean_T"],
            summary["mean_TSV"],
            yerr=summary["sem_TSV"],
            fmt="o-",
            capsize=4,
            label=gender
        )

    ax.set_title(walk_group)
    ax.set_xlabel("Temperature relative to walk mean (K)")
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Mean TSV")
axes[0].legend(title="Gender")
fig.suptitle("Mean TSV as a function of temperature, by gender", y=1.02)
plt.tight_layout()
plt.show()

# =========================================================
# 2) PLOT BY AGE GROUP
# =========================================================
age_order = ["<12", "13-15", "16-24", "25-54", "55-84"]



fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, walk_group in zip(axes, ["Cold walks", "Medium walks", "Hot walks"]):
    group_df = df_votes[df_votes["walk_group"] == walk_group].copy()

    for age in age_order:
        adf = group_df[group_df[age_col] == age].copy()
        adf = adf[["T_centered_walk_K", "TSV"]].dropna()

        if len(adf) == 0:
            continue

        # bin temperature
        adf["T_bin"] = pd.cut(adf["T_centered_walk_K"], bins=8)

        summary = (
            adf.groupby("T_bin", observed=False)
            .agg(
                mean_T=("T_centered_walk_K", "mean"),
                mean_TSV=("TSV", "mean"),
                std_TSV=("TSV", "std"),
                n=("TSV", "count")
            )
            .dropna()
            .reset_index()
            .sort_values("mean_T")
        )

        if len(summary) == 0:
            continue

        summary["sem_TSV"] = summary["std_TSV"] / np.sqrt(summary["n"])

        ax.errorbar(
            summary["mean_T"],
            summary["mean_TSV"],
            yerr=summary["sem_TSV"],
            fmt="o-",
            capsize=4,
            label=age
        )

    ax.set_title(walk_group)
    ax.set_xlabel("Temperature relative to walk mean (K)")
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Mean TSV")
axes[0].legend(title="Age group", bbox_to_anchor=(1.05, 1), loc="upper left")
fig.suptitle("Mean TSV as a function of temperature, by age group", y=1.02)
plt.tight_layout()
plt.show()

## I ara amb el TCV

In [ ]:

temp_col = "<T-T_fixed+<T>>"
y_col = "thermal_comfort"

tcv_map= {'Very comfortable': 0, 'Comfortable': 1, 'Slightly comfortable': 2, 'Neutral': 3, 'Slightly uncomfortable': 4, 'Uncomfortable': 5, 'Very uncomfortable': 6}

# ----------------------------
# Create walk and Kelvin temperature
# ----------------------------
df_votes["walk"] = df_votes["space_code"].astype(str).str[:2]
df_votes["T_K"] = df_votes[temp_col] + 273.15

# ----------------------------
# Temperature scaled by walk mean
# ----------------------------
df_votes["T_mean_walk_K"] = df_votes.groupby("walk")["T_K"].transform("mean")
df_votes["relative_temperature_walk_K"] = (
    (df_votes["T_K"] - df_votes["T_mean_walk_K"]) / df_votes["T_mean_walk_K"]
)

# ----------------------------
# Temperature scaled by global mean
# ----------------------------
T_mean_all_K = df_votes["T_K"].mean()
df_votes["relative_temperature_all_K"] = (
    (df_votes["T_K"] - T_mean_all_K) / T_mean_all_K
)

# ----------------------------
# Map TSV
# ----------------------------
df_votes["TSC"] = df_votes[y_col].map(tcv_map)

# ============================
# 1) Mean relative temperature by TSV class (scaled by walk)
# ============================
summary_walk = (
    df_votes[["relative_temperature_walk_K", "TSC"]]
    .dropna()
    .groupby("TSC", as_index=False)["relative_temperature_walk_K"]
    .agg(mean_T="mean", std_T="std", n="count")
    .sort_values("TSC")
)

summary_walk["sem_T"] = summary_walk["std_T"] / summary_walk["n"]**0.5

print("Scaled by walk mean:")
print(summary_walk)

plt.figure(figsize=(7, 5))
plt.errorbar(
    summary_walk["TSC"],
    summary_walk["mean_T"],
    yerr=summary_walk["sem_T"],   # use std_T instead if you want larger bars
    fmt="o",
    capsize=4
)

#plt.yticks([1, 2, 3, 4, 5, 6, 7],
     #      ["Slightly cool", "Cool", "Neutral", "Slightly warm", "Warm", "Hot", "Very hot"])
plt.xlabel("Mean relative temperature (scaled by walk mean, K)")
plt.ylabel("TSC")
plt.title("Mean relative temperature by TSC class (walk-scaled)")
plt.grid(True, alpha=0.3)
plt.show()

# ============================
# 2) Mean relative temperature by TSC class (scaled by all)
# ============================
summary_all = (
    df_votes[["relative_temperature_all_K", "TSC"]]
    .dropna()
    .groupby("TSC", as_index=False)["relative_temperature_all_K"]
    .agg(mean_T="mean", std_T="std", n="count")
    .sort_values("TSC")
)

summary_all["sem_T"] = summary_all["std_T"] / summary_all["n"]**0.5

print("Scaled by global mean:")
print(summary_all)

plt.figure(figsize=(7, 5))
plt.errorbar(
    summary_all["TSC"],
    summary_all["mean_T"],
    yerr=summary_all["sem_T"],   # use std_T instead if you want larger bars
    fmt="o",
    capsize=4
)

#plt.yticks([1, 2, 3, 4, 5, 6, 7],
  #         ["Slightly cool", "Cool", "Neutral", "Slightly warm", "Warm", "Hot", "Very hot"])
plt.xlabel("Mean relative temperature (scaled by global mean, K)")
plt.ylabel("TSV")
plt.title("Mean relative temperature by TSV class (global-scaled)")
plt.grid(True, alpha=0.3)
plt.show()